# README:
-The first three steps was to ensuring the data loaded properly.
-wdo file is located properly.
-All the updates inside wdo files are showing properly in the proper directory to run the code.

In [1]:
import subprocess
import pathlib

project_root = pathlib.Path(
    r"C:\Users\Nazmun\OneDrive - Midwestern State University\Documents\GitHub\4543-Spatial_mapping_new\04-Worldle Project"
)

print(f"Installing from: {project_root}")

result = subprocess.run(
    ["pip", "install", "-e", "."],
    cwd=project_root,
    capture_output=True,
    text=True,
)

print(result.stdout[-800:] if len(result.stdout) > 800 else result.stdout)

if result.returncode != 0:
    print("STDERR:", result.stderr[-400:])

Installing from: C:\Users\Nazmun\OneDrive - Midwestern State University\Documents\GitHub\4543-Spatial_mapping_new\04-Worldle Project
): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for wdo (pyproject.toml): started
  Building editable for wdo (pyproject.toml): finished with status 'done'
  Created wheel for wdo: filename=wdo-0.1.0-0.editable-py3-none-any.whl size=1225 sha256=960776812d21f91d9f43224e23ed7a2bb7f90c7a703d55d63c5f21b5681e92da
  Stored in directory: C:\Users\Nazmun\AppData\Local\Temp\pip-ephem-wheel-cache-g36st5pi\wheels\32\d6\57\58fcaeada1c0c1714e571a4e5c94b8eafc455f6108a304832c
Successfully built wdo
  Attempting uninstall: wdo
    Found existing installation: wdo 0.1.0
    Uninstalling wdo-0.1.0:
      Successfully uninstalled wdo-0.1.0



In [2]:
import os

# Set this to your project root (the folder that contains "wdo/")
project_path = r"C:\Users\Nazmun\OneDrive - Midwestern State University\Documents\GitHub\4543-Spatial_mapping_new\04-Worldle Project"

os.chdir(project_path)

# Confirm directory change
print("Current working directory:", os.getcwd())

# Now import
import wdo
print("wdo imported successfully")
print("wdo location:", wdo.__file__)

Current working directory: C:\Users\Nazmun\OneDrive - Midwestern State University\Documents\GitHub\4543-Spatial_mapping_new\04-Worldle Project
wdo imported successfully
wdo location: C:\Users\Nazmun\OneDrive - Midwestern State University\Documents\GitHub\4543-Spatial_mapping_new\04-Worldle Project\src\wdo\__init__.py


In [3]:
from pathlib import Path
cwd = Path.cwd()
print("Current Working Directory:")
print(cwd)

#parent = cwd.parent
#print(parent)

# add data folder to end of path
#data_path = cwd / "data"
data_path = cwd / "world_countries.json"
#data_path = parent_lib / "data"
print("\nLooking for data folder at:")
print(data_path)

#target = cwd / "data" / "world_countries.json"
target = cwd / "world_countries.json"
#target = parent / "countries.geojson"
print("\nAttempting to access:")
print(target)

exists = target.exists()
print("\nDoes file exist?")
print(exists)

assert exists, (
    f"\n❌ ERROR: File not found:\n{target}\n"
    "Check spelling and folder structure."
)

print("\n✅ File located successfully.")

Current Working Directory:
C:\Users\Nazmun\OneDrive - Midwestern State University\Documents\GitHub\4543-Spatial_mapping_new\04-Worldle Project

Looking for data folder at:
C:\Users\Nazmun\OneDrive - Midwestern State University\Documents\GitHub\4543-Spatial_mapping_new\04-Worldle Project\world_countries.json

Attempting to access:
C:\Users\Nazmun\OneDrive - Midwestern State University\Documents\GitHub\4543-Spatial_mapping_new\04-Worldle Project\world_countries.json

Does file exist?
True

✅ File located successfully.


In [ ]:
from ipyleaflet import Map, GeoJSON, CircleMarker
from wdo.io.geojson_tools import load_geojson
from pathlib import Path
from wdo.io.geojson_tools import load_geojson, iter_features, load_json
import os

# -----------------------------
# Load data
# -----------------------------
#countries = load_geojson("world_countries.json")

# -----------------------------
# 1. Load GeoJSON
# -----------------------------
countries_lookup = load_geojson("converted.geojson")
#map_country = load_geojson("countries.geojson")
# -----------------------------
# Bounding box center
# -----------------------------
def bbox_from_feature(feature):
    coords = feature["geometry"]["coordinates"]
    geom_type = feature["geometry"]["type"]

    points = []

    if geom_type == "Polygon":
        for ring in coords:
            points.extend(ring)

    elif geom_type == "MultiPolygon":
        for poly in coords:
            for ring in poly:
                points.extend(ring)

    else:
        return None

    lons = [p[0] for p in points]
    lats = [p[1] for p in points]

    return min(lons), min(lats), max(lons), max(lats)


def representative_point_bbox(feature):
    bbox = bbox_from_feature(feature)
    if bbox is None:
        return None

    min_lon, min_lat, max_lon, max_lat = bbox

    center_lat = (min_lat + max_lat) / 2
    center_lon = (min_lon + max_lon) / 2

    return center_lat, center_lon


# -----------------------------
# Create map
# -----------------------------
m = Map(center=(20, 0), zoom=2)

# -----------------------------
# Add all country polygons
# -----------------------------
geo_layer = GeoJSON(
    data=countries_lookup,
    style={
        "color": "#FF00FF", #magenta
        "weight": 1,
        "fillOpacity": 0.2
    }
)
m.add_layer(geo_layer)

# -----------------------------
# Add red center points
# -----------------------------
for feature in countries_lookup["features"]:
    center = representative_point_bbox(feature)

    if center is None:
        continue

    lat, lon = center

    marker = CircleMarker(
        location=(lat, lon),
        radius=3,
        color="#FF0000", #red
        fill_color="#FF0000", #red
        fill_opacity=0.2
    )

    m.add_layer(marker)

# -----------------------------
# Show map
# -----------------------------
m


Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

# First attempt to get the centroids. That looked like it was giving points outside the country borders, so I switched to using the bounding box centers instead. The red points on the map are the bounding box centers of each country.

In [5]:
# trying to use the wdo package to load the geojson file and find the center of the country. I am using the bounding box method to find the center of the country.
from ipyleaflet import CircleMarker
from wdo.io.geojson_tools import load_geojson, iter_features
from wdo.geometry.bbox import feature_center, largest_polygon_only  
from wdo.maps.leaflet_helpers import (
    make_map,
    add_geojson,
    fit_map_to_geojson
)

# -----------------------------
# Load data
# -----------------------------
countries_lookup = load_geojson("converted.geojson")


# -----------------------------
# Create map (WDO)
# -----------------------------
m = make_map(center=(20, 0), zoom=2)


# -----------------------------
# Add all country polygons
# -----------------------------
add_geojson(
    m,
    countries_lookup,
    name="Countries",
    style={
        "color": "#FF00FF",   # magenta
        "weight": 1,
        "fillOpacity": 0.2
    }
)


# -----------------------------
# Add red center points
# -----------------------------
for feature in iter_features(countries_lookup):

    center = feature_center(feature)   # ✅ reuse WDO

    if center is None:
        continue

    lat, lon = center

    marker = CircleMarker(
        location=(lat, lon),
        radius=3,
        color="#FF0000",
        fill_color="#FF0000",
        fill_opacity=0.7   # slightly more visible
    )

    m.add_layer(marker)


# -----------------------------
# Fit map to all countries
# -----------------------------
fit_map_to_geojson(m, countries_lookup)


# -----------------------------
# Show map
# -----------------------------
m
# Still looks like the center points are outside the country borders. I will need to investigate further to see if there is an issue with the feature_center function or if there is something else going on.

Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

# Still looks like the center points are outside the country borders. I will need to investigate further to see if there is an issue with the feature_center function or if there is something else going on.

In [6]:
#Trying more wdo functions to find the center of the country. I am using the largest polygon method to find the center of the country.
from ipyleaflet import CircleMarker
from wdo.io.geojson_tools import load_geojson, iter_features
from wdo.geometry.bbox import feature_center, largest_polygon_only
from wdo.geometry.bearing import initial_bearing, bearing_to_compass
from wdo.maps.leaflet_helpers import (
    make_map,
    add_geojson,
    add_path,
    fit_map_to_geojson
)

# -----------------------------
# Load data
# -----------------------------
countries_lookup = load_geojson("converted.geojson")


# -----------------------------
# Create map (WDO)
# -----------------------------
m = make_map(center=(20, 0), zoom=2)


# -----------------------------
# Add country polygons
# -----------------------------
add_geojson(
    m,
    countries_lookup,
    name="Countries",
    style={
        "color": "#FF00FF",   # magenta
        "weight": 1,
        "fillOpacity": 0.2
    }
)


# -----------------------------
# Add center points (FIXED)
# -----------------------------
centers = []

for feature in iter_features(countries_lookup):

    # ✅ FIX: clean MultiPolygon countries
    feature_clean = largest_polygon_only(feature)

    center = feature_center(feature_clean)

    if center is None:
        continue

    lat, lon = center
    centers.append((lat, lon))

    marker = CircleMarker(
        location=(lat, lon),
        radius=3,
        color="#FF0000",
        fill_color="#FF0000",
        fill_opacity=0.8
    )

    m.add_layer(marker)


# -----------------------------
# OPTIONAL: Draw bearings between random pairs (debug/learning)
# -----------------------------
# This helps visually verify direction logic

for i in range(0, min(10, len(centers) - 1)):
    p1 = centers[i]
    p2 = centers[i + 1]

    b = initial_bearing(p1, p2)
    direction = bearing_to_compass(b)

    # Debug print
    print(f"{i}: {direction} ({b:.1f}°)")

    # Draw small line
    add_path(
        m,
        [p1, p2],
        color="#00FFFF",  # cyan
        weight=1
    )


# -----------------------------
# Fit map to all countries
# -----------------------------
fit_map_to_geojson(m, countries_lookup)


# -----------------------------
# Show map
# -----------------------------
m

0: W (291.6°)
1: S (171.2°)
2: N (21.7°)
3: NW (301.1°)
4: S (161.5°)
5: NE (63.9°)
6: E (93.1°)
7: NE (46.1°)
8: W (286.4°)
9: N (15.0°)


Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

In [7]:
from ipyleaflet import CircleMarker
from wdo.io.geojson_tools import load_geojson, iter_features
from wdo.geometry.bbox import feature_center, largest_polygon_only
from wdo.geometry.bearing import initial_bearing, bearing_to_compass
from wdo.maps.leaflet_helpers import (
    make_map,
    add_geojson,
    add_path,
    fit_map_to_geojson
)

# -----------------------------
# Load data
# -----------------------------
countries_lookup = load_geojson("converted.geojson")


# -----------------------------
# Create map (WDO)
# -----------------------------
m = make_map(center=(20, 0), zoom=2)


# -----------------------------
# Add country polygons
# -----------------------------
add_geojson(
    m,
    countries_lookup,
    name="Countries",
    style={
        "color": "#FF00FF",   # magenta
        "weight": 1,
        "fillOpacity": 0.2
    }
)


# -----------------------------
# Add center points (FIXED)
# -----------------------------
centers = []

for feature in iter_features(countries_lookup):

    # ✅ FIX: clean MultiPolygon countries
    feature_clean = largest_polygon_only(feature)

    center = feature_center(feature_clean)

    if center is None:
        continue

    lat, lon = center
    centers.append((lat, lon))

    marker = CircleMarker(
        location=(lat, lon),
        radius=3,
        color="#FF0000",
        fill_color="#FF0000",
        fill_opacity=0.2
    )

    m.add_layer(marker)


# -----------------------------

# -----------------------------
# Fit map to all countries
# -----------------------------
fit_map_to_geojson(m, countries_lookup)


# -----------------------------
# Show map
# -----------------------------
m

Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

# I was able to get the centers of countries by not behaving strange.

In [8]:
from ipyleaflet import CircleMarker
from wdo.io.geojson_tools import load_geojson, iter_features
from wdo.geometry.bbox import feature_center, largest_polygon_only
from wdo.maps.leaflet_helpers import (
    make_map,
    add_geojson,
    fit_map_to_geojson
)

# -----------------------------
# Load data
# -----------------------------
countries = load_geojson("converted.geojson")


# -----------------------------
# Create map (WDO)
# -----------------------------
m = make_map(center=(20, 0), zoom=2)


# -----------------------------
# Add country polygons
# -----------------------------
add_geojson(
    m,
    countries,
    name="Countries",
    style={
        "color": "#FF00FF",   # magenta
        "weight": 1,
        "fillOpacity": 0.2
    }
)


# -----------------------------
# Compute + plot centers
# -----------------------------
centers = []

for feature in iter_features(countries):

    # Fix MultiPolygon distortion (USA, Russia, etc.)
    clean_feature = largest_polygon_only(feature)

    center = feature_center(clean_feature)

    if center is None:
        continue

    lat, lon = center
    centers.append((lat, lon))

    m.add_layer(
        CircleMarker(
            location=(lat, lon),
            radius=3,
            color="#FF0000",
            fill_color="#FF0000",
            fill_opacity=0.7
        )
    )


# -----------------------------
# Fit map to all countries
# -----------------------------
fit_map_to_geojson(m, countries)


# -----------------------------
# Output
# -----------------------------
print(f"Total centers plotted: {len(centers)}")

m

Total centers plotted: 219


Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

# The next goal is to get the bbox center and mean center for the countries.

In [19]:
def _extract_points(feature):
    """Return a flat list of (lon, lat) points from Polygon/MultiPolygon."""
    geom = feature.get("geometry", {})
    coords = geom.get("coordinates")
    geom_type = geom.get("type")

    if coords is None:
        return []

    points = []

    if geom_type == "Polygon":
        for ring in coords:
            points.extend(ring)

    elif geom_type == "MultiPolygon":
        for poly in coords:
            for ring in poly:
                points.extend(ring)

    else:
        raise ValueError(f"Unsupported geometry type: {geom_type}")

    return points


def feature_center(feature, method="bbox"):
    """Return a representative (lat, lon) for a Polygon/MultiPolygon feature.

    method="bbox"  -> center of the bounding box (fast, mostly fine)
    method="mean"  -> mean of boundary vertices (alternative; try both)

    Why we expose the method: Worldle judges "direction from your guess to the target."
    If your centers are garbage, your arrows are garbage. Students should eyeball
    their centers on a map before trusting them.
    """
    points = _extract_points(feature)

    if not points:
        raise ValueError("No coordinates found in feature")

    if method == "bbox":
        lons = [p[0] for p in points]
        lats = [p[1] for p in points]

        min_lon, max_lon = min(lons), max(lons)
        min_lat, max_lat = min(lats), max(lats)

        center_lat = (min_lat + max_lat) / 2
        center_lon = (min_lon + max_lon) / 2

        return center_lat, center_lon

    elif method == "mean":
        avg_lon = sum(p[0] for p in points) / len(points)
        avg_lat = sum(p[1] for p in points) / len(points)

        return avg_lat, avg_lon

    else:
        raise ValueError(f"Unknown method: {method}")
    
def get_country_feature(code):
    # Case 1: FeatureCollection
    if "features" in countries_lookup:
        for f in countries_lookup["features"]:
            props = f.get("properties", {})
            if (
                props.get("ISO3166-1-Alpha-3") == code or
                props.get("ISO_A3") == code
            ):
                return f

    # Case 2: Lookup dictionary
    if isinstance(countries_lookup, dict):
        return countries_lookup.get(code)

    return None

# Create map
# -----------------------------
m = Map(center=(20, 0), zoom=2)

# -----------------------------
# Add all country polygons
# -----------------------------
geo_layer = GeoJSON(
    data=countries_lookup,
    style={
        "color": "#FF00FF", #magenta
        "weight": 1,
        "fillOpacity": 0.2
    }
)
m.add_layer(geo_layer)

# -----------------------------
# Add red center points
# -----------------------------
for feature in countries_lookup["features"]:
    center = representative_point_bbox(feature)

    if center is None:
        continue

    lat, lon = center

    marker = CircleMarker(
        location=(lat, lon),
        radius=3,
        color="#FF0000", #red
        fill_color="#FF0000", #red
        fill_opacity=0.2
    )

    m.add_layer(marker)

# -----------------------------
# Show map
# -----------------------------
m



Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

In [21]:
Chile = get_country_feature("CHL")
Algeria = get_country_feature("DZA")
Russia = get_country_feature("RUS")
Nauru = get_country_feature("NRU")
Spain = get_country_feature("ESP")
United_States = get_country_feature("USA")

print("Spain BBox center:", feature_center(Spain, method="bbox"))
print("Spain Mean center:", feature_center(Spain, method="mean"))
print("United States BBox center:", feature_center(United_States, method="bbox"))
print("United States Mean center:", feature_center(United_States, method="mean"))

print("Algeria BBox center:", feature_center(Algeria, method="bbox"))
print("Algeria Mean center:", feature_center(Algeria, method="mean"))
print("Chile BBox center:", feature_center(Chile, method="bbox"))
print("Chile Mean center:", feature_center(Chile, method="mean"))
print("Russia BBox center:", feature_center(Russia, method="bbox"))
print("Russia Mean center:", feature_center(Russia, method="mean"))
print("Nauru BBox center:", feature_center(Nauru, method="bbox"))
print("Nauru Mean center:", feature_center(Nauru, method="mean"))

Spain BBox center: (35.717841, -6.9150694999999995)
Spain Mean center: (39.05785932202821, -5.122569351914036)
United States BBox center: (45.1593095, 0.3187159999999949)
United States Mean center: (48.05246205897557, -120.5187521641144)
Algeria BBox center: (28.0347505, 1.6432380000000002)
Algeria Mean center: (30.77757573098479, 2.8141908582866293)
Chile BBox center: (-36.712546, -87.9372655)
Chile Mean center: (-47.20757422277141, -72.85661663365704)
Russia BBox center: (61.5256955, 0.0)
Russia Mean center: (63.694453661307, 86.4109668302318)
Nauru BBox center: (-0.521132, 166.932628)
Nauru Mean center: (-0.5177818888888889, 166.93537677777778)


## writing codes to get the center of the country using the wdo package. I am using the feature_center_mean function to get the center of the country. I am also using the largest_polygon_only function to get the largest polygon of the country to get a more accurate center point. I am also using the add_geojson function

In [9]:
import wdo.geometry.bbox as b

print(dir(b))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_extract_points', 'bbox_from_feature', 'bbox_from_features', 'bbox_from_points', 'bbox_to_polygon', 'feature_center', 'feature_center_mean', 'fix_antimeridian', 'largest_polygon_only']


In [22]:
from ipyleaflet import CircleMarker
from wdo.io.geojson_tools import load_geojson, iter_features
from wdo.geometry.bbox import largest_polygon_only, feature_center_mean
from wdo.maps.leaflet_helpers import make_map, add_geojson, fit_map_to_geojson

# -----------------------------
# Load data
# -----------------------------
countries = load_geojson("converted.geojson")


# -----------------------------
# Create map (WDO)
# -----------------------------
m = make_map(center=(20, 0), zoom=2)


# -----------------------------
# Add polygons
# -----------------------------
add_geojson(
    m,
    countries,
    name="Countries",
    style={
        "color": "#FF00FF",
        "weight": 1,
        "fillOpacity": 0.2
    }
)


# -----------------------------
# Add MEAN centers
# -----------------------------
centers = []

for feature in iter_features(countries):

    # Fix MultiPolygon distortion (IMPORTANT)
    clean = largest_polygon_only(feature)

    center = feature_center_mean(clean)

    if center is None:
        continue

    lat, lon = center
    centers.append((lat, lon))

    m.add_layer(
        CircleMarker(
            location=(lat, lon),
            radius=3,
            color="#ff0000",          # red for mean center
            fill_color="#ff0000",
            fill_opacity=0.2
        )
    )

def get_country_feature(code):
    # Case 1: FeatureCollection
    if "features" in countries_lookup:
        for f in countries_lookup["features"]:
            props = f.get("properties", {})
            if (
                props.get("ISO3166-1-Alpha-3") == code or
                props.get("ISO_A3") == code
            ):
                return f

    # Case 2: Lookup dictionary
    if isinstance(countries_lookup, dict):
        return countries_lookup.get(code)

    return None

# -----------------------------
# Fit map
# -----------------------------
fit_map_to_geojson(m, countries)

print(f"Mean centers plotted: {len(centers)}")

m

Mean centers plotted: 219


Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

In [15]:
from ipyleaflet import CircleMarker
from wdo.io.geojson_tools import load_geojson, iter_features
from wdo.geometry.bbox import feature_center_mean, largest_polygon_only
#from wdo.geometry.mean_center import feature_center_mean
from wdo.maps.leaflet_helpers import make_map, add_geojson, fit_map_to_geojson

# -----------------------------
# Load data
# -----------------------------
countries = load_geojson("converted.geojson")


# -----------------------------
# Create map
# -----------------------------
m = make_map(center=(20, 0), zoom=2)


# -----------------------------
# Add polygons
# -----------------------------
add_geojson(
    m,
    countries,
    name="Countries",
    style={
        "color": "#FF00FF",
        "weight": 1,
        "fillOpacity": 0.2
    }
)


# -----------------------------
# Compare centers
# -----------------------------
bbox_centers = []
mean_centers = []

for feature in iter_features(countries):

    # Fix MultiPolygon issue
    clean = largest_polygon_only(feature)

    # --- BBOX CENTER ---
    bbox_center = feature_center(clean)

    # --- MEAN CENTER ---
    mean_center = feature_center_mean(clean)

    if not bbox_center or not mean_center:
        continue

    lat_b, lon_b = bbox_center
    lat_m, lon_m = mean_center

    bbox_centers.append((lat_b, lon_b))
    mean_centers.append((lat_m, lon_m))

    # 🔴 Plot BBOX center (RED)
    m.add_layer(
        CircleMarker(
            location=(lat_b, lon_b),
            radius=3,
            color="#FF0000",
            fill_color="#FF0000",
            fill_opacity=0.8
        )
    )

    # 🔵 Plot MEAN center (BLUE)
    m.add_layer(
        CircleMarker(
            location=(lat_m, lon_m),
            radius=3,
            color="#0000FF",
            fill_color="#0000FF",
            fill_opacity=0.8
        )
    )


# -----------------------------
# Print example (Algeria)
# -----------------------------
def get_country_feature(data, code):
    for f in iter_features(data):
        props = f.get("properties", {})
        if props.get("ISO3166-1-Alpha-3") == code or props.get("ISO_A3") == code:
            return f
    return None


algeria = get_country_feature(countries, "DZA")

if algeria:
    clean = largest_polygon_only(algeria)

    print("Algeria BBox center:", feature_center(clean))
    print("Algeria Mean center:", feature_center_mean(clean))


# -----------------------------
# Fit map
# -----------------------------
fit_map_to_geojson(m, countries)

print(f"BBox centers plotted: {len(bbox_centers)}")
print(f"Mean centers plotted: {len(mean_centers)}")

m

Algeria BBox center: (28.0347505, 1.6432380000000002)
Algeria Mean center: (30.77757573098479, 2.8141908582866293)
BBox centers plotted: 219
Mean centers plotted: 219


Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…


# Finally I was able to get the bbox center and mean center calculated.

In [17]:
united_states = get_country_feature(countries, "USA")
clean = largest_polygon_only(united_states)
print("United States BBox center:", feature_center(clean))
print("United States Mean center:", feature_center_mean(clean))

United States BBox center: (37.2454495, -95.855966)
United States Mean center: (36.56148564510196, -91.15230081607356)


In [18]:
Chile = get_country_feature(countries, "CHL")
clean = largest_polygon_only(Chile)
print("Chile BBox center:", feature_center(clean))
print("Chile Mean center:", feature_center_mean(clean)) 

Chile BBox center: (-35.696294, -71.362691)
Chile Mean center: (-42.78537781840626, -72.4518213453573)
